# Weekly Submission Preview

**Author:** Jan  
**Last updated:** 2026-05-19

Sanity-check the production output before mailing the CSV. Set `TARGET_DATE` to the upcoming Friday, run all cells, then run `python -m src.submit TARGET_DATE` to actually generate the file.

## Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import PROFESSOR_EMAIL, TEAM_ID
from src.fundamental import BASELINE_WEIGHT, MAX_TILT, UNIVERSE, get_weights
from src.submit import build_submission_row

pd.options.display.float_format = "{:.4f}".format

## Configure the target Friday

In [ ]:
TARGET_DATE = "2026-05-29"   # update each Friday
target = pd.Timestamp(TARGET_DATE)
assert target.day_name() == "Friday", f"{TARGET_DATE} is a {target.day_name()}, not a Friday"
print(f"Submission for Friday {TARGET_DATE} (effective week starting {(target + pd.Timedelta(days=3)).date()})")

## Model state on the target date

In [ ]:
weights = get_weights(target)
gld_tilt = weights["GLD"] - BASELINE_WEIGHT

print(f"GLD tilt from baseline:   {gld_tilt:+.4f}  (max possible: +/- {MAX_TILT:.3f})")
print()
print("Target weights:")
print(weights.round(4))
print()
print(f"Sum check (should be 1.0): {weights.sum():.6f}")
assert abs(weights.sum() - 1.0) < 1e-9, "weights don't sum to 1"
assert ((weights >= 0) & (weights <= 1)).all(), "weights out of [0, 1]"
assert list(weights.index) == UNIVERSE, f"schema mismatch: {list(weights.index)}"

## Visualisation

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["steelblue", "lightgreen", "goldenrod", "lightcoral"]
bars = ax.bar(weights.index, weights.values, color=colors, alpha=0.85, edgecolor="white")
ax.axhline(BASELINE_WEIGHT, color="gray", linestyle="--", linewidth=1, label=f"Equal weight ({BASELINE_WEIGHT})")
for bar, w in zip(bars, weights.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{w*100:.2f}%", ha="center", va="bottom", fontsize=10)
ax.set_ylim(0, max(0.5, weights.max() * 1.15))
ax.set_ylabel("Target weight")
ax.set_title(f"Portfolio for week of {TARGET_DATE}")
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## CSV preview (what the submission file will contain)

In [ ]:
row = build_submission_row(target, team_id=TEAM_ID)
print("week,team_id,acwi,agg,gld,bsv")
print(f"{row['week']},{row['team_id']},{row['acwi']:.2f},{row['agg']:.2f},{row['gld']:.2f},{row['bsv']:.2f}")
print()
post_round_sum = row['acwi'] + row['agg'] + row['gld'] + row['bsv']
print(f"Sum of rounded weights: {post_round_sum:.2f} (should be 100.00)")

## Email metadata

In [ ]:
team_number = TEAM_ID.replace("Team", "")
print(f"Recipient:     {PROFESSOR_EMAIL}")
print(f"Subject:       Algorithmic Trading Project | Team {team_number} | Portfolio for Week {TARGET_DATE}")
print(f"Attachment:    submissions/{TEAM_ID}_{TARGET_DATE}.csv  (run `python -m src.submit {TARGET_DATE}` to generate)")

## Pre-flight checklist

Before hitting send:

- [ ] `TEAM_ID` in `src/config.py` is set to your real team number (currently `"TeamXX"` placeholder)
- [ ] `TARGET_DATE` above matches the Friday you're submitting
- [ ] Sum of rounded weights = 100.00 (cell above)
- [ ] You ran `python -m src.submit <TARGET_DATE>` and the CSV file exists in `submissions/`
- [ ] Email subject and recipient match the spec from `Project Guidelines - Group.pdf` §3.6
- [ ] (Weeks 2-5 only) total allocation change vs last week's submission ≤ 25pp